
# AHMET ÇALIK VAKFI – CASE STUDY 02  
## Notebook 03: Veri Hikâyesi, Müşteri Segmentasyonu ve İş Önerileri

Bu notebook, ilk iki notebookta hazırlanan temizlenmiş verileri iş kararına dönüştürür.

Ana sorular:

1. İlçeler arasındaki tüketim farkının temel nedeni nedir?
2. Mevsimsellik hangi ilçelerde daha belirgindir?
3. Yüksek tüketimli müşteriler kimlerdir?
4. Hangi müşteriler yüksek finansal değer taşırken aynı zamanda tahsilat riski oluşturmaktadır?
5. Müşteri grupları için hangi aksiyonlar önerilebilir?

> Not: “Yüksek tüketim”, “yüksek finansal değer” ve “VIP müşteri” aynı kavram değildir.  
> Finansal değer için tahakkuk tutarı, risk için ödeme zamanlaması ve kalan bakiye birlikte değerlendirilir.


In [11]:

# ======================================================================
# 1. KÜTÜPHANELER VE AYARLAR
# ======================================================================

import os
import re
import warnings
import unicodedata

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("=" * 80)
print("NOTEBOOK 03 - VERİ HİKÂYESİ VE İŞ ANALİZİ")
print("=" * 80)


NOTEBOOK 03 - VERİ HİKÂYESİ VE İŞ ANALİZİ



> **Önemli yöntem notu**
>
> Bu notebookta müşteri segmentasyonu yalnızca ilk 50 müşteriden oluşturulmaz.  
> `musteri_bazli_tuketim_ozeti.csv` dosyası tüm müşterileri içermelidir.
>
> Ayrıca dosya bulunamadığında sentetik ödeme verisi üretilmez. Gerçek veri yoksa notebook hata vererek durur. Böylece sonuçlara yapay kayıt karışmaz.


## 1. Veri kaynaklarının yüklenmesi

In [2]:

# ======================================================================
# 2. DOSYA YOLLARI
# ======================================================================

TAHAKKUK_YOLU = (
    r"C:\Users\ÇaY GüZeLi\Downloads\notebook_01_ciktilari"
    r"\tahakkuk_isaretli_tam_veri.csv"
)

MUSTERI_OZET_YOLU = (
    r"C:\Users\ÇaY GüZeLi\Downloads\notebook_01_ciktilari"
    r"\musteri_bazli_tuketim_ozeti.csv"
)

TAHSILAT_OZET_YOLU = r"C:\veri_2.csv"

CIKTI_KLASORU = os.path.join(os.getcwd(), "notebook_03_ciktilari")
os.makedirs(CIKTI_KLASORU, exist_ok=True)

print(f"Çıktı klasörü: {CIKTI_KLASORU}")


# ======================================================================
# 3. YARDIMCI FONKSİYONLAR
# ======================================================================

def oku(path):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Dosya bulunamadı: {path}")

    return pd.read_csv(
        path,
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        low_memory=False
    )


def turkce_karakterleri_duzelt(metin):
    ceviri = str.maketrans({
        "ı": "i", "İ": "i",
        "ğ": "g", "Ğ": "g",
        "ü": "u", "Ü": "u",
        "ş": "s", "Ş": "s",
        "ö": "o", "Ö": "o",
        "ç": "c", "Ç": "c"
    })
    return str(metin).translate(ceviri)


def temiz_sutun_adi(sutun):
    sutun = str(sutun).replace("\ufeff", "").strip().casefold()
    sutun = turkce_karakterleri_duzelt(sutun)
    sutun = unicodedata.normalize("NFKD", sutun)
    sutun = "".join(
        karakter for karakter in sutun
        if not unicodedata.combining(karakter)
    )
    sutun = re.sub(r"[^a-z0-9]+", "_", sutun)
    sutun = re.sub(r"_+", "_", sutun).strip("_")
    return sutun


def standartlastir(df):
    df = df.copy()
    df.columns = [temiz_sutun_adi(col) for col in df.columns]

    yeniden_adlandirma = {
        "i_l": "il",
        "i_lce": "ilce",
        "soz_hsp_bagimsiz": "sozlesme_hesap_no",
        "soz_hsp_bagimsiz_": "sozlesme_hesap_no"
    }

    df = df.rename(
        columns={
            eski: yeni
            for eski, yeni in yeniden_adlandirma.items()
            if eski in df.columns
        }
    )

    return df


def guvenli_bol(pay, payda):
    payda = payda.replace(0, np.nan)
    return pay / payda


def yuzdelik_etiket(series):
    return series.rank(pct=True, method="average") * 100


def ilk_gecerli(series):
    seri = series.dropna()
    return seri.iloc[0] if not seri.empty else np.nan


# ======================================================================
# 4. VERİLERİ OKU
# ======================================================================

df = standartlastir(oku(TAHAKKUK_YOLU))
musteri_tuketim = standartlastir(oku(MUSTERI_OZET_YOLU))
df_odeme = standartlastir(oku(TAHSILAT_OZET_YOLU))

for veri in [df, musteri_tuketim, df_odeme]:
    if "sozlesme_hesap_no" in veri.columns:
        veri["sozlesme_hesap_no"] = (
            veri["sozlesme_hesap_no"]
            .astype("string")
            .str.replace(r"\.0$", "", regex=True)
            .str.strip()
        )

if "kwh" in df.columns:
    df["kwh"] = pd.to_numeric(df["kwh"], errors="coerce")

if "fatura_tarihi" in df.columns:
    df["fatura_tarihi"] = pd.to_datetime(df["fatura_tarihi"], errors="coerce")
    df["ay"] = df["fatura_tarihi"].dt.month
    df["yil"] = df["fatura_tarihi"].dt.year
elif "mali_yil_donem" in df.columns:
    df["mali_yil_donem"] = pd.to_datetime(df["mali_yil_donem"], errors="coerce")
    df["ay"] = df["mali_yil_donem"].dt.month
    df["yil"] = df["mali_yil_donem"].dt.year

print(f" Tahakkuk verisi       : {len(df):,} satır")
print(f" Müşteri tüketim özeti : {len(musteri_tuketim):,} müşteri")
print(f" Tahsilat özet verisi  : {len(df_odeme):,} satır")


Çıktı klasörü: C:\Users\ÇaY GüZeLi\Downloads\notebook_03_ciktilari
✓ Tahakkuk verisi       : 1,185,698 satır
✓ Müşteri tüketim özeti : 28,290 müşteri
✓ Tahsilat özet verisi  : 917,632 satır


## 2. İlçeler arasındaki tüketim farkının analizi

In [3]:

# ======================================================================
# 5. İLÇE BAZINDA TÜKETİM HİKÂYESİ
# ======================================================================

ilce_ozet = (
    df[df["kwh"].notna()]
    .groupby("ilce")
    .agg(
        toplam_tuketim_kwh=("kwh", "sum"),
        ortalama_kwh=("kwh", "mean"),
        medyan_kwh=("kwh", "median"),
        standart_sapma_kwh=("kwh", "std"),
        kayit_sayisi=("kwh", "count"),
        benzersiz_musteri=("sozlesme_hesap_no", "nunique")
    )
    .reset_index()
)

ilce_ozet["musteri_basina_toplam_tuketim_kwh"] = (
    ilce_ozet["toplam_tuketim_kwh"] /
    ilce_ozet["benzersiz_musteri"].replace(0, np.nan)
)

ilce_ozet["ortalama_medyan_orani"] = (
    ilce_ozet["ortalama_kwh"] /
    ilce_ozet["medyan_kwh"].replace(0, np.nan)
)

ilce_ozet = ilce_ozet.sort_values(
    "toplam_tuketim_kwh",
    ascending=False
)

display(ilce_ozet)

en_yuksek_toplam = ilce_ozet.iloc[0]
en_dusuk_toplam = ilce_ozet.iloc[-1]

display(Markdown(
    f"""
### İlçe bazlı temel bulgu

- Toplam tüketimi en yüksek ilçe **{en_yuksek_toplam['ilce']}** olmuştur.
- Toplam tüketimi en düşük ilçe **{en_dusuk_toplam['ilce']}** olmuştur.
- Ancak toplam tüketim, müşteri sayısından doğrudan etkilenir. Bu nedenle yalnızca toplam kWh ile karar verilmemelidir.
- Müşteri başına tüketim ve kayıt başına ortalama tüketim birlikte değerlendirilmelidir.
- Ortalama/medyan oranının yüksek olması, az sayıda büyük tüketicinin ilçe ortalamasını yukarı çektiğine işaret eder.
"""
))


,ilce,toplam_tuketim_kwh,ortalama_kwh,medyan_kwh,standart_sapma_kwh,kayit_sayisi,benzersiz_musteri,musteri_basina_toplam_tuketim_kwh,ortalama_medyan_orani
1,GÜMÜŞHACIKÖY,"74,526,473.58",97.34,48.31,"1,077.76",765657,18190,"4,097.11",2.01
0,GÖYNÜCEK,"26,472,614.19",89.67,45.09,742.28,295223,7128,"3,713.89",1.99
2,HAMAMÖZÜ,"8,846,428.21",70.87,40.56,389.22,124818,2981,"2,967.60",1.75



### İlçe bazlı temel bulgu

- Toplam tüketimi en yüksek ilçe **GÜMÜŞHACIKÖY** olmuştur.
- Toplam tüketimi en düşük ilçe **HAMAMÖZÜ** olmuştur.
- Ancak toplam tüketim, müşteri sayısından doğrudan etkilenir. Bu nedenle yalnızca toplam kWh ile karar verilmemelidir.
- Müşteri başına tüketim ve kayıt başına ortalama tüketim birlikte değerlendirilmelidir.
- Ortalama/medyan oranının yüksek olması, az sayıda büyük tüketicinin ilçe ortalamasını yukarı çektiğine işaret eder.


### Hesap sınıfı kompozisyonu

In [4]:

# ======================================================================
# 6. İLÇE FARKLARININ HESAP SINIFI KOMPOZİSYONU İLE AÇIKLANMASI
# ======================================================================

sinif_ilce = (
    df.groupby(["ilce", "hesap_sinifi"], as_index=False)
    .agg(
        toplam_kwh=("kwh", "sum"),
        ortalama_kwh=("kwh", "mean"),
        medyan_kwh=("kwh", "median"),
        benzersiz_musteri=("sozlesme_hesap_no", "nunique")
    )
)

sinif_ilce["ilce_toplam_kwh"] = (
    sinif_ilce.groupby("ilce")["toplam_kwh"].transform("sum")
)

sinif_ilce["ilce_tuketim_payi_yuzde"] = (
    sinif_ilce["toplam_kwh"] /
    sinif_ilce["ilce_toplam_kwh"].replace(0, np.nan) * 100
)

dominant_siniflar = (
    sinif_ilce.sort_values(
        ["ilce", "ilce_tuketim_payi_yuzde"],
        ascending=[True, False]
    )
    .groupby("ilce")
    .head(5)
)

display(dominant_siniflar)

for ilce, grup in dominant_siniflar.groupby("ilce"):
    birinci = grup.iloc[0]
    display(Markdown(
        f"""
**{ilce}:** Toplam tüketimde en yüksek paya sahip hesap sınıfı  
**{birinci['hesap_sinifi']}** ve payı yaklaşık **%{birinci['ilce_tuketim_payi_yuzde']:.1f}** seviyesindedir.
"""
    ))

print(
    "Yorum: İlçeler arasındaki fark yalnızca müşteri sayısından değil, "
    "ticari, tarımsal, kamu veya sanayi tipi müşterilerin portföy içindeki ağırlığından da kaynaklanabilir."
)


,ilce,hesap_sinifi,toplam_kwh,ortalama_kwh,medyan_kwh,benzersiz_musteri,ilce_toplam_kwh,ilce_tuketim_payi_yuzde
8,GÖYNÜCEK,Mesken,"14,031,370.69",54.44,45.37,6217,"26,472,614.19",53.00
20,GÖYNÜCEK,Ticari Faaliyet - Yazıhane,"3,104,339.47",153.45,39.64,463,"26,472,614.19",11.73
18,GÖYNÜCEK,Tarımsal Faaliyetler (Kooperatif),"2,864,914.30","3,515.23",171.75,31,"26,472,614.19",10.82
19,GÖYNÜCEK,Tarımsal Faaliyetler (Şahıs),"2,302,245.91",879.39,21.76,66,"26,472,614.19",8.70
6,GÖYNÜCEK,Köy İçme Suyu Temini ve Dağıtımı Tesisi,"926,655.25",710.08,263.12,37,"26,472,614.19",3.50
44,GÜMÜŞHACIKÖY,Mesken,"37,674,622.87",57.24,49.39,15592,"74,526,473.58",50.55
55,GÜMÜŞHACIKÖY,Ticari Faaliyet - Yazıhane,"10,605,553.97",166.75,42.30,1504,"74,526,473.58",14.23
54,GÜMÜŞHACIKÖY,Tarımsal Faaliyetler (Şahıs),"7,025,620.78",502.30,15.96,354,"74,526,473.58",9.43
29,GÜMÜŞHACIKÖY,1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,"4,323,094.30",892.46,24.44,140,"74,526,473.58",5.80
53,GÜMÜŞHACIKÖY,Tarımsal Faaliyetler (Kooperatif),"3,290,488.93","4,097.74",374.40,24,"74,526,473.58",4.42



**GÖYNÜCEK:** Toplam tüketimde en yüksek paya sahip hesap sınıfı  
**Mesken** ve payı yaklaşık **%53.0** seviyesindedir.



**GÜMÜŞHACIKÖY:** Toplam tüketimde en yüksek paya sahip hesap sınıfı  
**Mesken** ve payı yaklaşık **%50.6** seviyesindedir.



**HAMAMÖZÜ:** Toplam tüketimde en yüksek paya sahip hesap sınıfı  
**Mesken** ve payı yaklaşık **%60.6** seviyesindedir.


Yorum: İlçeler arasındaki fark yalnızca müşteri sayısından değil, ticari, tarımsal, kamu veya sanayi tipi müşterilerin portföy içindeki ağırlığından da kaynaklanabilir.


## 3. Mevsimsellik ve tüketim davranışı

In [5]:

# ======================================================================
# 7. AYLIK MEVSİMSELLİK
# ======================================================================

ay_adlari = {
    1: "Ocak", 2: "Şubat", 3: "Mart", 4: "Nisan",
    5: "Mayıs", 6: "Haziran", 7: "Temmuz", 8: "Ağustos",
    9: "Eylül", 10: "Ekim", 11: "Kasım", 12: "Aralık"
}

aylik_ilce = (
    df[df["kwh"] >= 0]
    .groupby(["ilce", "ay"], as_index=False)
    .agg(
        ortalama_kwh=("kwh", "mean"),
        medyan_kwh=("kwh", "median"),
        toplam_kwh=("kwh", "sum"),
        kayit_sayisi=("kwh", "count")
    )
)

zirve_aylar = (
    aylik_ilce.sort_values(
        ["ilce", "ortalama_kwh"],
        ascending=[True, False]
    )
    .groupby("ilce")
    .head(1)
    .copy()
)

zirve_aylar["ay_adi"] = zirve_aylar["ay"].map(ay_adlari)

display(zirve_aylar[
    ["ilce", "ay_adi", "ortalama_kwh", "medyan_kwh", "toplam_kwh"]
])

for _, satir in zirve_aylar.iterrows():
    display(Markdown(
        f"""
- **{satir['ilce']}** için en yüksek ortalama tüketim **{satir['ay_adi']}** ayında görülmüştür.
  Bu durum soğutma, tarımsal sulama, mevsimsel ticari faaliyet veya belirli büyük tüketicilerle ilişkili olabilir.
"""
    ))

print(
    "Not: Mevsimsel artışın nedeni kesin olarak söylenmeden önce hesap sınıfı bazında aylık kırılım kontrol edilmelidir."
)


,ilce,ay_adi,ortalama_kwh,medyan_kwh,toplam_kwh
6,GÖYNÜCEK,Temmuz,128.49,52.36,"4,187,027.21"
18,GÜMÜŞHACIKÖY,Temmuz,130.04,46.89,"10,722,206.72"
24,HAMAMÖZÜ,Ocak,90.53,44.75,"775,666.25"



- **GÖYNÜCEK** için en yüksek ortalama tüketim **Temmuz** ayında görülmüştür.
  Bu durum soğutma, tarımsal sulama, mevsimsel ticari faaliyet veya belirli büyük tüketicilerle ilişkili olabilir.



- **GÜMÜŞHACIKÖY** için en yüksek ortalama tüketim **Temmuz** ayında görülmüştür.
  Bu durum soğutma, tarımsal sulama, mevsimsel ticari faaliyet veya belirli büyük tüketicilerle ilişkili olabilir.



- **HAMAMÖZÜ** için en yüksek ortalama tüketim **Ocak** ayında görülmüştür.
  Bu durum soğutma, tarımsal sulama, mevsimsel ticari faaliyet veya belirli büyük tüketicilerle ilişkili olabilir.


Not: Mevsimsel artışın nedeni kesin olarak söylenmeden önce hesap sınıfı bazında aylık kırılım kontrol edilmelidir.


## 4. Ödeme davranışının hazırlanması

In [6]:

# ======================================================================
# 8. ÖDEME ZAMANLAMA DEĞİŞKENLERİ
# ======================================================================

odeme_gruplari = {
    "erken_tutar": [
        "son_odeme_tarihinden_onceki_tahsilat"
    ],
    "vadesinde_tutar": [
        "son_odeme_tarihindeki_tahsilat"
    ],
    "gecikme_1_5_tutar": [
        "son_odeme_1",
        "son_odeme_2",
        "son_odeme_3",
        "son_odeme_4",
        "son_odeme_5"
    ],
    "gecikme_6_30_tutar": [
        "son_odeme_6_10",
        "son_odeme_10_20",
        "son_odeme_20_30"
    ],
    "gecikme_31_90_tutar": [
        "son_odeme_30_60",
        "son_odeme_60_90"
    ],
    "gecikme_91_180_tutar": [
        "son_odeme_90_120",
        "son_odeme_120_150",
        "son_odeme_150_180"
    ],
    "gecikme_180_ustu_tutar": [
        "son_odeme_180"
    ]
}

sayisal_odeme_sutunlari = ["tahakkuk_tutar"]

for sutunlar in odeme_gruplari.values():
    sayisal_odeme_sutunlari.extend(sutunlar)

sayisal_odeme_sutunlari = [
    col for col in sayisal_odeme_sutunlari
    if col in df_odeme.columns
]

for col in sayisal_odeme_sutunlari:
    df_odeme[col] = pd.to_numeric(
        df_odeme[col],
        errors="coerce"
    )

for yeni_sutun, eski_sutunlar in odeme_gruplari.items():
    mevcutlar = [
        col for col in eski_sutunlar
        if col in df_odeme.columns
    ]

    if mevcutlar:
        df_odeme[yeni_sutun] = (
            df_odeme[mevcutlar]
            .fillna(0)
            .sum(axis=1)
        )
    else:
        df_odeme[yeni_sutun] = 0.0

df_odeme["zamaninda_tutar"] = (
    df_odeme["erken_tutar"] +
    df_odeme["vadesinde_tutar"]
)

df_odeme["gecikmeli_tutar"] = (
    df_odeme["gecikme_1_5_tutar"] +
    df_odeme["gecikme_6_30_tutar"] +
    df_odeme["gecikme_31_90_tutar"] +
    df_odeme["gecikme_91_180_tutar"] +
    df_odeme["gecikme_180_ustu_tutar"]
)

df_odeme["toplam_tahsil_edilen_tutar"] = (
    df_odeme["zamaninda_tutar"] +
    df_odeme["gecikmeli_tutar"]
)

df_odeme["kalan_bakiye"] = (
    df_odeme["tahakkuk_tutar"].fillna(0) -
    df_odeme["toplam_tahsil_edilen_tutar"].fillna(0)
).clip(lower=0)

df_odeme["zamaninda_odeme_orani"] = guvenli_bol(
    df_odeme["zamaninda_tutar"],
    df_odeme["tahakkuk_tutar"]
).clip(lower=0, upper=1)

df_odeme["gecikmeli_odeme_orani"] = guvenli_bol(
    df_odeme["gecikmeli_tutar"],
    df_odeme["tahakkuk_tutar"]
).clip(lower=0, upper=1)

df_odeme["bakiye_orani"] = guvenli_bol(
    df_odeme["kalan_bakiye"],
    df_odeme["tahakkuk_tutar"]
).clip(lower=0, upper=1)

print(" Ödeme zamanlaması ve bakiye değişkenleri oluşturuldu.")


✓ Ödeme zamanlaması ve bakiye değişkenleri oluşturuldu.


## 5. Müşteri bazında finansal değer ve risk

In [7]:

# ======================================================================
# 9. MÜŞTERİ BAZINDA ÖDEME ÖZETİ
# ======================================================================

musteri_odeme = (
    df_odeme.groupby("sozlesme_hesap_no", as_index=False)
    .agg(
        toplam_tahakkuk_tutar=("tahakkuk_tutar", "sum"),
        zamaninda_tutar=("zamaninda_tutar", "sum"),
        gecikmeli_tutar=("gecikmeli_tutar", "sum"),
        gecikme_31_90_tutar=("gecikme_31_90_tutar", "sum"),
        gecikme_91_180_tutar=("gecikme_91_180_tutar", "sum"),
        gecikme_180_ustu_tutar=("gecikme_180_ustu_tutar", "sum"),
        kalan_bakiye=("kalan_bakiye", "sum"),
        fatura_sayisi=("tahakkuk_tutar", "count"),
        ilce_odeme=("ilce", ilk_gecerli),
        hesap_sinifi_odeme=("hesap_sinifi", ilk_gecerli)
    )
)

musteri_odeme["zamaninda_odeme_orani"] = guvenli_bol(
    musteri_odeme["zamaninda_tutar"],
    musteri_odeme["toplam_tahakkuk_tutar"]
).clip(lower=0, upper=1)

musteri_odeme["gecikmeli_odeme_orani"] = guvenli_bol(
    musteri_odeme["gecikmeli_tutar"],
    musteri_odeme["toplam_tahakkuk_tutar"]
).clip(lower=0, upper=1)

musteri_odeme["bakiye_orani"] = guvenli_bol(
    musteri_odeme["kalan_bakiye"],
    musteri_odeme["toplam_tahakkuk_tutar"]
).clip(lower=0, upper=1)

musteri_odeme["agir_gecikme_orani"] = guvenli_bol(
    (
        musteri_odeme["gecikme_91_180_tutar"] +
        musteri_odeme["gecikme_180_ustu_tutar"]
    ),
    musteri_odeme["toplam_tahakkuk_tutar"]
).clip(lower=0, upper=1)

# Tüketim ve ödeme özetlerini birleştir
musteri = musteri_tuketim.merge(
    musteri_odeme,
    on="sozlesme_hesap_no",
    how="outer"
)

if "ilce" not in musteri.columns and "ilce_odeme" in musteri.columns:
    musteri["ilce"] = musteri["ilce_odeme"]
elif "ilce" in musteri.columns and "ilce_odeme" in musteri.columns:
    musteri["ilce"] = musteri["ilce"].fillna(musteri["ilce_odeme"])

if "hesap_sinifi" not in musteri.columns and "hesap_sinifi_odeme" in musteri.columns:
    musteri["hesap_sinifi"] = musteri["hesap_sinifi_odeme"]
elif "hesap_sinifi" in musteri.columns and "hesap_sinifi_odeme" in musteri.columns:
    musteri["hesap_sinifi"] = musteri["hesap_sinifi"].fillna(
        musteri["hesap_sinifi_odeme"]
    )

# Eksik finansal oranları sıfır kabul etmek yerine ayrı tut
oran_sutunlari = [
    "zamaninda_odeme_orani",
    "gecikmeli_odeme_orani",
    "bakiye_orani",
    "agir_gecikme_orani"
]

# Değer skoru: tahakkuk tutarı öncelikli, tüketim büyüklüğü destekleyici
musteri["tahakkuk_yuzdelik"] = yuzdelik_etiket(
    musteri["toplam_tahakkuk_tutar"].fillna(0)
)

musteri["tuketim_yuzdelik"] = yuzdelik_etiket(
    musteri["toplam_tuketim_kwh"].fillna(0)
)

musteri["deger_skoru"] = (
    0.70 * musteri["tahakkuk_yuzdelik"] +
    0.30 * musteri["tuketim_yuzdelik"]
)

# Açıklanabilir risk skoru
musteri["risk_skoru"] = (
    100 * (
        0.40 * musteri["bakiye_orani"].fillna(0) +
        0.30 * musteri["gecikmeli_odeme_orani"].fillna(0) +
        0.20 * musteri["agir_gecikme_orani"].fillna(0) +
        0.10 * (1 - musteri["zamaninda_odeme_orani"].fillna(0))
    )
).clip(0, 100)

musteri["risk_seviyesi"] = pd.cut(
    musteri["risk_skoru"],
    bins=[-0.01, 25, 50, 75, 100],
    labels=["Düşük", "Orta", "Yüksek", "Kritik"]
)

deger_esigi = musteri["deger_skoru"].quantile(0.80)
risk_esigi = 50

musteri["musteri_segmenti"] = np.select(
    [
        (musteri["deger_skoru"] >= deger_esigi) &
        (musteri["risk_skoru"] < risk_esigi),

        (musteri["deger_skoru"] >= deger_esigi) &
        (musteri["risk_skoru"] >= risk_esigi),

        (musteri["deger_skoru"] < deger_esigi) &
        (musteri["risk_skoru"] < risk_esigi),

        (musteri["deger_skoru"] < deger_esigi) &
        (musteri["risk_skoru"] >= risk_esigi)
    ],
    [
        "Yüksek değer - düzenli",
        "Yüksek değer - riskli",
        "Standart değer - düzenli",
        "Standart değer - riskli"
    ],
    default="İncelenecek"
)

display(musteri[
    [
        "sozlesme_hesap_no",
        "ilce",
        "hesap_sinifi",
        "toplam_tuketim_kwh",
        "toplam_tahakkuk_tutar",
        "zamaninda_odeme_orani",
        "gecikmeli_odeme_orani",
        "bakiye_orani",
        "deger_skoru",
        "risk_skoru",
        "risk_seviyesi",
        "musteri_segmenti"
    ]
].sort_values(
    ["deger_skoru", "risk_skoru"],
    ascending=[False, False]
).head(30))


,sozlesme_hesap_no,ilce,hesap_sinifi,toplam_tuketim_kwh,toplam_tahakkuk_tutar,zamaninda_odeme_orani,gecikmeli_odeme_orani,bakiye_orani,deger_skoru,risk_skoru,risk_seviyesi,musteri_segmenti
17108,6414845714,GÜMÜŞHACIKÖY,Lisansız Üreticiler,"2,634,973.74","8,174,792.53",0.12,0.88,0.00,100.00,35.09,Orta,Yüksek değer - düzenli
20635,7553584057,GÜMÜŞHACIKÖY,1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,"1,636,590.30","7,045,969.22",0.94,0.06,0.00,100.00,2.55,Düşük,Yüksek değer - düzenli
23307,8398443615,GÜMÜŞHACIKÖY,Karayolları Genel Müdürlüğü Aydınlatma,"1,238,340.68","4,923,408.69",0.61,0.39,0.00,99.99,16.10,Düşük,Yüksek değer - düzenli
12091,4831215580,GÜMÜŞHACIKÖY,İçme-Kullanma Suyu (Belediye),"771,460.44","3,964,506.19",0.00,1.00,0.00,99.99,40.68,Orta,Yüksek değer - düzenli
8857,3798287663,GÜMÜŞHACIKÖY,Belediye,"782,153.22","3,814,784.30",0.00,1.00,0.00,99.99,40.50,Orta,Yüksek değer - düzenli
2027,16419436,GÜMÜŞHACIKÖY,Ticari Faaliyet - Yazıhane,"776,755.31","3,550,594.24",1.00,0.00,0.00,99.99,0.00,Düşük,Yüksek değer - düzenli
11914,4777252213,GÜMÜŞHACIKÖY,1 SAYILI CETVELDE YER ALAN KAMU İDARESİ,"857,108.70","3,421,065.10",0.96,0.04,0.00,99.99,1.49,Düşük,Yüksek değer - düzenli
14286,5515560446,GÖYNÜCEK,İçme-Kullanma Suyu (Belediye),"566,836.64","2,561,139.14",0.18,0.82,0.00,99.98,35.41,Orta,Yüksek değer - düzenli
23451,8449517062,GÜMÜŞHACIKÖY,"Resmi SAĞLIK KURULUŞLARI,RESMİ SPOR TES.","443,369.50","2,017,916.68",0.65,0.35,0.00,99.98,14.09,Düşük,Yüksek değer - düzenli
10876,4429981104,GÜMÜŞHACIKÖY,Aritma Tesisleri,"458,395.92","1,918,206.28",0.43,0.57,0.00,99.98,22.92,Düşük,Yüksek değer - düzenli


## 6. Segmentlerin iş açısından yorumlanması

In [8]:

# ======================================================================
# 10. SEGMENT ÖZETİ VE İŞ AKSİYONLARI
# ======================================================================

segment_ozet = (
    musteri.groupby("musteri_segmenti", dropna=False)
    .agg(
        musteri_sayisi=("sozlesme_hesap_no", "nunique"),
        toplam_tuketim_kwh=("toplam_tuketim_kwh", "sum"),
        toplam_tahakkuk_tutar=("toplam_tahakkuk_tutar", "sum"),
        toplam_kalan_bakiye=("kalan_bakiye", "sum"),
        ortalama_risk_skoru=("risk_skoru", "mean"),
        medyan_risk_skoru=("risk_skoru", "median"),
        ortalama_deger_skoru=("deger_skoru", "mean")
    )
    .reset_index()
)

toplam_musteri = segment_ozet["musteri_sayisi"].sum()
toplam_tahakkuk = segment_ozet["toplam_tahakkuk_tutar"].sum()
toplam_bakiye = segment_ozet["toplam_kalan_bakiye"].sum()

segment_ozet["musteri_payi_yuzde"] = (
    segment_ozet["musteri_sayisi"] /
    toplam_musteri * 100
)

segment_ozet["tahakkuk_payi_yuzde"] = (
    segment_ozet["toplam_tahakkuk_tutar"] /
    toplam_tahakkuk * 100
)

segment_ozet["bakiye_payi_yuzde"] = (
    segment_ozet["toplam_kalan_bakiye"] /
    toplam_bakiye * 100
)

display(segment_ozet.sort_values(
    "toplam_tahakkuk_tutar",
    ascending=False
))

aksiyonlar = {
    "Yüksek değer - düzenli": (
        "Sadakat ve koruma stratejisi uygulanmalıdır. Otomatik ödeme, dijital fatura, "
        "enerji verimliliği danışmanlığı ve kişiselleştirilmiş hizmetler önerilebilir."
    ),
    "Yüksek değer - riskli": (
        "Tahsilat ekipleri için öncelikli gruptur. Erken uyarı, ödeme planı, "
        "taksitlendirme, teminat ve kredi limiti değerlendirmesi yapılmalıdır."
    ),
    "Standart değer - düzenli": (
        "Düşük maliyetli dijital iletişim kanalları kullanılmalı, otomatik ödeme "
        "ve e-fatura kullanımına yönlendirilmelidir."
    ),
    "Standart değer - riskli": (
        "Operasyon maliyeti kontrollü tutulmalıdır. Otomatik hatırlatma, SMS ve "
        "dijital tahsilat kanalları önceliklendirilmelidir."
    )
}

for segment, metin in aksiyonlar.items():
    satir = segment_ozet[
        segment_ozet["musteri_segmenti"] == segment
    ]

    if not satir.empty:
        satir = satir.iloc[0]

        display(Markdown(
            f"""
### {segment}

- Müşteri sayısı: **{satir['musteri_sayisi']:,.0f}**
- Toplam tahakkuk içindeki pay: **%{satir['tahakkuk_payi_yuzde']:.1f}**
- Toplam bakiye içindeki pay: **%{satir['bakiye_payi_yuzde']:.1f}**
- Önerilen aksiyon: {metin}
"""
        ))


,musteri_segmenti,musteri_sayisi,toplam_tuketim_kwh,toplam_tahakkuk_tutar,toplam_kalan_bakiye,ortalama_risk_skoru,medyan_risk_skoru,ortalama_deger_skoru,musteri_payi_yuzde,tahakkuk_payi_yuzde,bakiye_payi_yuzde
2,Yüksek değer - düzenli,10649,"88,479,073.96","348,562,800.41","185,122.44",10.96,4.87,84.17,19.99,74.69,48.17
0,Standart değer - düzenli,41631,"21,254,982.31","117,743,461.17","178,157.12",12.65,7.18,42.04,78.16,25.23,46.36
3,Yüksek değer - riskli,5,"77,324.43","240,346.53",0.00,52.25,51.91,90.86,0.01,0.05,0.00
1,Standart değer - riskli,981,"34,135.28","142,098.04","20,998.81",53.97,51.97,16.65,1.84,0.03,5.46



### Yüksek değer - düzenli

- Müşteri sayısı: **10,649**
- Toplam tahakkuk içindeki pay: **%74.7**
- Toplam bakiye içindeki pay: **%48.2**
- Önerilen aksiyon: Sadakat ve koruma stratejisi uygulanmalıdır. Otomatik ödeme, dijital fatura, enerji verimliliği danışmanlığı ve kişiselleştirilmiş hizmetler önerilebilir.



### Yüksek değer - riskli

- Müşteri sayısı: **5**
- Toplam tahakkuk içindeki pay: **%0.1**
- Toplam bakiye içindeki pay: **%0.0**
- Önerilen aksiyon: Tahsilat ekipleri için öncelikli gruptur. Erken uyarı, ödeme planı, taksitlendirme, teminat ve kredi limiti değerlendirmesi yapılmalıdır.



### Standart değer - düzenli

- Müşteri sayısı: **41,631**
- Toplam tahakkuk içindeki pay: **%25.2**
- Toplam bakiye içindeki pay: **%46.4**
- Önerilen aksiyon: Düşük maliyetli dijital iletişim kanalları kullanılmalı, otomatik ödeme ve e-fatura kullanımına yönlendirilmelidir.



### Standart değer - riskli

- Müşteri sayısı: **981**
- Toplam tahakkuk içindeki pay: **%0.0**
- Toplam bakiye içindeki pay: **%5.5**
- Önerilen aksiyon: Operasyon maliyeti kontrollü tutulmalıdır. Otomatik hatırlatma, SMS ve dijital tahsilat kanalları önceliklendirilmelidir.


## 7. Yönetici özeti

In [9]:

# ======================================================================
# 11. OTOMATİK YÖNETİCİ ÖZETİ
# ======================================================================

en_yuksek_ilce = ilce_ozet.iloc[0]["ilce"]

riskli_yuksek_deger = musteri[
    musteri["musteri_segmenti"] == "Yüksek değer - riskli"
].copy()

riskli_bakiye = riskli_yuksek_deger["kalan_bakiye"].sum()
riskli_tahakkuk = riskli_yuksek_deger["toplam_tahakkuk_tutar"].sum()
riskli_musteri_sayisi = riskli_yuksek_deger["sozlesme_hesap_no"].nunique()

en_riskli_ilce = (
    musteri.groupby("ilce", dropna=False)
    .agg(
        ortalama_risk_skoru=("risk_skoru", "mean"),
        toplam_bakiye=("kalan_bakiye", "sum"),
        musteri_sayisi=("sozlesme_hesap_no", "nunique")
    )
    .sort_values("toplam_bakiye", ascending=False)
    .reset_index()
)

display(en_riskli_ilce)

if not en_riskli_ilce.empty:
    bakiye_yuksek_ilce = en_riskli_ilce.iloc[0]["ilce"]
else:
    bakiye_yuksek_ilce = "Belirlenemedi"

display(Markdown(
    f"""
# Yönetici Özeti

1. **Tüketim yapısı:** Toplam tüketimi en yüksek ilçe **{en_yuksek_ilce}** olarak belirlenmiştir. 
   Ancak bu sonuç müşteri sayısı ve hesap sınıfı kompozisyonu ile birlikte değerlendirilmelidir.

2. **Mevsimsellik:** İlçelerin zirve tüketim ayları aynı olmayabilir. Yaz aylarındaki artışlar özellikle 
   tarımsal sulama, soğutma yükleri ve ticari faaliyet yoğunluğu ile ilişkili olabilir.

3. **Yüksek değerli müşteri grubu:** Müşteri değeri yalnızca tüketimle değil, toplam tahakkuk tutarıyla birlikte değerlendirilmiştir. 
   Bu nedenle yüksek kWh kullanan her müşteri otomatik olarak VIP kabul edilmemiştir.

4. **Tahsilat riski:** **{riskli_musteri_sayisi:,.0f}** müşteri “Yüksek değer - riskli” segmentinde yer almaktadır. 
   Bu grubun toplam tahakkuk tutarı **{riskli_tahakkuk:,.2f} TL**, kalan bakiyesi ise **{riskli_bakiye:,.2f} TL** seviyesindedir.

5. **Bölgesel risk:** Toplam kalan bakiyenin en yüksek olduğu ilçe **{bakiye_yuksek_ilce}** olarak belirlenmiştir.

6. **Öneri:** Tahsilat ekipleri müşteri sayısına göre değil, “yüksek değer + yüksek risk” kombinasyonuna göre önceliklendirilmelidir. 
   Düzenli ödeme yapan yüksek değerli müşteriler ise sadakat ve koruma programlarına alınmalıdır.
"""
))


,ilce,ortalama_risk_skoru,toplam_bakiye,musteri_sayisi
0,TAŞOVA,13.20,"169,064.37",24702
1,GÖYNÜCEK,14.40,"113,770.69",7197
2,GÜMÜŞHACIKÖY,12.29,"77,584.39",18363
3,HAMAMÖZÜ,13.74,"23,858.92",3004



# Yönetici Özeti

1. **Tüketim yapısı:** Toplam tüketimi en yüksek ilçe **GÜMÜŞHACIKÖY** olarak belirlenmiştir. 
   Ancak bu sonuç müşteri sayısı ve hesap sınıfı kompozisyonu ile birlikte değerlendirilmelidir.

2. **Mevsimsellik:** İlçelerin zirve tüketim ayları aynı olmayabilir. Yaz aylarındaki artışlar özellikle 
   tarımsal sulama, soğutma yükleri ve ticari faaliyet yoğunluğu ile ilişkili olabilir.

3. **Yüksek değerli müşteri grubu:** Müşteri değeri yalnızca tüketimle değil, toplam tahakkuk tutarıyla birlikte değerlendirilmiştir. 
   Bu nedenle yüksek kWh kullanan her müşteri otomatik olarak VIP kabul edilmemiştir.

4. **Tahsilat riski:** **5** müşteri “Yüksek değer - riskli” segmentinde yer almaktadır. 
   Bu grubun toplam tahakkuk tutarı **240,346.53 TL**, kalan bakiyesi ise **0.00 TL** seviyesindedir.

5. **Bölgesel risk:** Toplam kalan bakiyenin en yüksek olduğu ilçe **TAŞOVA** olarak belirlenmiştir.

6. **Öneri:** Tahsilat ekipleri müşteri sayısına göre değil, “yüksek değer + yüksek risk” kombinasyonuna göre önceliklendirilmelidir. 
   Düzenli ödeme yapan yüksek değerli müşteriler ise sadakat ve koruma programlarına alınmalıdır.


## 8. Sonuç tablolarının kaydedilmesi

In [10]:

# ======================================================================
# 12. ÇIKTILARI KAYDET
# ======================================================================

ciktilar = {
    "ilce_tuketim_ozeti.csv": ilce_ozet,
    "ilce_hesap_sinifi_analizi.csv": sinif_ilce,
    "aylik_ilce_tuketim_analizi.csv": aylik_ilce,
    "musteri_deger_risk_segmentasyonu.csv": musteri,
    "segment_ozeti.csv": segment_ozet,
    "ilce_risk_ozeti.csv": en_riskli_ilce,
    "yuksek_deger_riskli_musteriler.csv": riskli_yuksek_deger
}

for dosya_adi, veri in ciktilar.items():
    veri.to_csv(
        os.path.join(CIKTI_KLASORU, dosya_adi),
        sep=";",
        decimal=",",
        encoding="utf-8-sig",
        index=False
    )
    print(f" {dosya_adi:<45} {len(veri):>10,} satır")

excel_yolu = os.path.join(
    CIKTI_KLASORU,
    "notebook_03_veri_hikayesi_raporu.xlsx"
)

try:
    with pd.ExcelWriter(excel_yolu, engine="openpyxl") as writer:
        ilce_ozet.to_excel(
            writer,
            sheet_name="Ilce_Tuketim_Ozeti",
            index=False
        )

        dominant_siniflar.to_excel(
            writer,
            sheet_name="Ilce_Dominant_Siniflar",
            index=False
        )

        zirve_aylar.to_excel(
            writer,
            sheet_name="Zirve_Tuketim_Aylari",
            index=False
        )

        segment_ozet.to_excel(
            writer,
            sheet_name="Segment_Ozeti",
            index=False
        )

        en_riskli_ilce.to_excel(
            writer,
            sheet_name="Ilce_Risk_Ozeti",
            index=False
        )

        riskli_yuksek_deger.sort_values(
            "risk_skoru",
            ascending=False
        ).head(1000).to_excel(
            writer,
            sheet_name="Yuksek_Deger_Riskli",
            index=False
        )

    print(f"\n Excel raporu oluşturuldu: {excel_yolu}")

except ImportError:
    print(
        "[UYARI] openpyxl bulunamadığı için Excel raporu oluşturulamadı. "
        "Kurulum: pip install openpyxl"
    )
except Exception as hata:
    print(f"[UYARI] Excel raporu oluşturulamadı: {hata}")

print("\n" + "=" * 80)
print("NOTEBOOK 03 BAŞARIYLA TAMAMLANDI")
print("=" * 80)
print(f"Çıktı klasörü: {CIKTI_KLASORU}")


✓ ilce_tuketim_ozeti.csv                                 3 satır
✓ ilce_hesap_sinifi_analizi.csv                         93 satır
✓ aylik_ilce_tuketim_analizi.csv                        36 satır
✓ musteri_deger_risk_segmentasyonu.csv              53,266 satır
✓ segment_ozeti.csv                                      4 satır
✓ ilce_risk_ozeti.csv                                    4 satır
✓ yuksek_deger_riskli_musteriler.csv                     5 satır

✓ Excel raporu oluşturuldu: C:\Users\ÇaY GüZeLi\Downloads\notebook_03_ciktilari\notebook_03_veri_hikayesi_raporu.xlsx

NOTEBOOK 03 BAŞARIYLA TAMAMLANDI
Çıktı klasörü: C:\Users\ÇaY GüZeLi\Downloads\notebook_03_ciktilari
